# 00 — Installation and first network


This notebook introduces the pure-Python `nucnetpy` package and builds a tiny reaction network directly in Python.

The tutorial assumes this folder is inside the package repository. From a terminal, install the package first with:

```bash
cd nucnetpy_pure_python
python -m pip install -e .[dev,hdf5]
```

Inside Jupyter, the setup cell below also adds the local `src/` directory to `sys.path`, so the notebooks can run before installation.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)

## Build a toy network

We use an intentionally simple alpha-chain example:

\[
{}^4\mathrm{He} + {}^4\mathrm{He} ightarrow {}^8\mathrm{Be},\qquad
{}^8\mathrm{Be} + {}^4\mathrm{He} ightarrow {}^{12}\mathrm{C}.
\]

The rate constants are artificial. They are chosen only to demonstrate the package interface.

In [ ]:
def build_toy_alpha_network():
    """Build a small alpha-chain toy network for tutorials.

    This is not intended to be a physical helium-burning network.  The rates are
    deliberately simple constants so that the examples run quickly and are easy
    to inspect.
    """
    net = nn.Network()
    for name in ["he4", "be8", "c12"]:
        net.add_species(nn.Species.parse(name))

    r1 = nn.Reaction.from_names(
        ["he4", "he4"], ["be8"],
        constant_rate=2.0e-6,
        q_value=0.092,
        label="toy_2alpha_to_be8",
    )
    r2 = nn.Reaction.from_names(
        ["be8", "he4"], ["c12"],
        constant_rate=5.0e-5,
        q_value=7.367,
        label="toy_be8_alpha_to_c12",
    )
    net.reactions.add(r1)
    net.reactions.add(r2)

    zone = nn.Zone(label=("toy", "0", "0"), properties={"t9": "1.0", "rho": "1e4"})
    zone.set_abundance("he4", 0.25)  # X(he4) = A*Y = 1.0
    zone.set_abundance("be8", 0.0)
    zone.set_abundance("c12", 0.0)
    net.add_zone(zone)
    return net

net = build_toy_alpha_network()
print(net)
print('species:', net.species_names())
print('number of reactions:', len(net.reactions.reactions))
print('number of zones:', len(net.zones))

In [ ]:
for r in net.reactions.reactions:
    print(r.label, ':', r.string, 'Q=', r.q_value)

## Inspect the initial zone

`Y` is abundance per baryon. The mass fraction is `X_i = A_i Y_i`.

In [ ]:
zone = net.zone(0)
print('zone label:', zone.label)
print('properties:', zone.properties)
print('Y:', zone.abundances)
print('X:', zone.mass_fractions(net.species))
print('sum X =', zone.xsum(net.species), 'Ye =', zone.ye(net.species))